# Chaînes-02 — La règle de Born, l'unitarité et la relation d'incertitude

Deuxième carnet de la sous-série **Chaînes** : la physique fondamentale lue comme des réseaux de paires intriquées. Le premier carnet ([Chaînes-01](Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb)) a suivi la chaîne complète de l'axiome au candidat $\alpha$ ; celui-ci descend d'un cran et distille les chapitres 40 et 41 du **compendium Chain Theory** (A. D. Wissner-Gross, dépôt [`physical-superintelligence/chain-theory`](https://github.com/physical-superintelligence/chain-theory)) — les trois résultats que la littérature appelle règle de Born, unitarité et relation d'incertitude.

Ce carnet est le **jumeau C#** de [Chaînes-02 — version Python](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb) :
mêmes entrées, mêmes valeurs, et une arithmétique **entièrement exacte** — rationnels réduits et matrices de Pauli
sur `System.Numerics.BigInteger`, complexe exact (partie réelle et partie imaginaire rationnelles) et anneau
cyclotomique `Z[ω]` pour les lettres de Weyl en dimension 3. Aucune égalité flottante n'est testée : chaque
contrôle lève sur écart (`InvalidOperationException`), là où le jumeau Python compare des tolérances.

**Statut de la source, à lire avant tout le reste.** Le compendium est un **preprint non relu** qui se déclare lui-même *working draft*. Les trois résultats de ce carnet y sont des **théorèmes conditionnels** — voici le résumé de la source, mot pour mot : *« From the axiom we obtain, as conditional theorems, the state space, the Born rule, unitarity, the uncertainty relation, the rotation and Lorentz groups, the signature of spacetime, and parity, time reversal, and CPT, and, as a theorem, conservation laws. »* La release tient le compte : 100 théorèmes, 109 théorèmes conditionnels, 198 résultats ouverts, et 64 des 100 théorèmes liés à des déclarations Lean nommées. Les deux documents sont archivés au gisement (licence CC BY 4.0) :

- Letter : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2026 - Wissner-Gross - Chain Theory - The fine structure constant from a single axiom (letter).pdf`
- Compendium : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2026 - Wissner-Gross - Chain Theory Compendium (v0.1 working draft).pdf`

**Ce que ce carnet fait** : il rejoue lui-même les parties calculables des deux chapitres — l'équivalence du théorème 40.1 dans les deux sens (état réduit plat $\iff$ matrice de coefficients multiple d'une unitaire sur le support, éq. 51), le groupe d'appariement et l'envariance $(U \otimes \bar U)|\omega\rangle = |\omega\rangle$, le canon de Born $\mathrm{Tr}(\rho P)$ aux observables d'étiquette (éq. 56), le join de la figure 15 comme unitaire sur son support, la relation de Weyl aux dimensions 2, 3 et 4, le recensement des paires de lettres non commutantes (3, 24 et 84) et des paires de bases distinctes (3, 6 et 21), et la borne de Robertson aux sept registres du voisinage (éq. 57) avec sa saturation mesurée.

**Ce qu'il n'est pas** : une validation de l'axiome ni des prémices. Chaque théorème est *conditionnel sur le texte de l'axiome et sur les résultats des chapitres nommés comme prémices, chacun à son propre rang* — parmi eux des **résultats ouverts** (2.4, 2.5, 2.7, 2.8, 18.5, 18.6). Rien ici ne les transforme en résultats établis ; ce que la source ne prouve pas est listé ligne à ligne en fin de carnet.

| Section | Contenu |
|---|---|
| 1 | Le cadre épistémique — Mesuré, Cité, Ouvert ; la graduation de la source |
| 2 | La paire, sa matrice de coefficients, l'état réduit |
| 3 | Théorème 40.1 : le pli plat et l'unitarité du transport, du join et du pas |
| 4 | Le groupe d'appariement et l'envariance |
| 5 | La règle de Born : le canon $\mathrm{Tr}(\rho P)$ |
| 6 | Le join de la figure 15 comme unitaire sur son support |
| 7 | Les lettres de Weyl et leur relation aux dimensions 2, 3, 4 |
| 8 | Les bases du lien, les observables d'étiquette et l'équation 55 |
| 9 | La borne de Robertson aux sept registres, équation 57 ; le recensement des paires de bases |
| 10 | Ce que la source ne prouve pas ; croisements avec le dépôt |
| 11 | Exercices |

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

// Tout le carnet tourne en arithmetique exacte : System.Numerics.BigInteger pour les
// entiers, un rationnel exact reduit (numerateur et denominateur entiers) la ou la
// source emploie fractions.Fraction, un complexe exact (partie reelle et partie
// imaginaire rationnelles) et l'anneau cyclotomique Z[omega] pour les lettres de Weyl
// en dimension 3. Aucune egalite flottante n'est testee : chaque controle leve sur
// ecart (InvalidOperationException).
Console.WriteLine(".NET " + Environment.Version + " ; arithmetique exacte BigInteger, aucune egalite flottante testee");

public static class VerifC02
{
    public static void Egale(bool condition, string message)
    {
        if (!condition) throw new InvalidOperationException("ECART : " + message);
    }
}

public readonly struct RationnelC02 : IEquatable<RationnelC02>
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }

    public RationnelC02(BigInteger numerateur) : this(numerateur, BigInteger.One) { }

    public RationnelC02(BigInteger numerateur, BigInteger denominateur)
    {
        if (denominateur == BigInteger.Zero)
            throw new DivideByZeroException("Denominateur nul : rationnel indefini.");
        if (denominateur < BigInteger.Zero)
        {
            numerateur = -numerateur;
            denominateur = -denominateur;
        }
        BigInteger g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(numerateur), denominateur);
        if (g == BigInteger.Zero) g = BigInteger.One;
        Numerateur = numerateur / g;
        Denominateur = denominateur / g;
    }

    public static readonly RationnelC02 Zero = new RationnelC02(BigInteger.Zero);
    public static readonly RationnelC02 Un = new RationnelC02(BigInteger.One);

    public static RationnelC02 operator +(RationnelC02 a, RationnelC02 b) =>
        new RationnelC02(a.Numerateur * b.Denominateur + b.Numerateur * a.Denominateur,
                        a.Denominateur * b.Denominateur);
    public static RationnelC02 operator -(RationnelC02 a, RationnelC02 b) =>
        new RationnelC02(a.Numerateur * b.Denominateur - b.Numerateur * a.Denominateur,
                        a.Denominateur * b.Denominateur);
    public static RationnelC02 operator *(RationnelC02 a, RationnelC02 b) =>
        new RationnelC02(a.Numerateur * b.Numerateur, a.Denominateur * b.Denominateur);
    public static RationnelC02 operator *(RationnelC02 a, BigInteger k) =>
        new RationnelC02(k * a.Numerateur, a.Denominateur);
    public static RationnelC02 operator *(BigInteger k, RationnelC02 a) => a * k;
    public static RationnelC02 operator /(RationnelC02 a, RationnelC02 b) =>
        new RationnelC02(a.Numerateur * b.Denominateur, a.Denominateur * b.Numerateur);
    public static RationnelC02 operator -(RationnelC02 a) =>
        new RationnelC02(-a.Numerateur, a.Denominateur);

    public static bool operator ==(RationnelC02 a, RationnelC02 b) =>
        a.Numerateur == b.Numerateur && a.Denominateur == b.Denominateur;
    public static bool operator !=(RationnelC02 a, RationnelC02 b) => !(a == b);

    public bool Equals(RationnelC02 autre) => this == autre;
    public override bool Equals(object? obj) => obj is RationnelC02 r && Equals(r);
    public override int GetHashCode() => HashCode.Combine(Numerateur, Denominateur);

    // Ecriture decimale EXACTE quand elle existe (denominateur = 2^a 5^b), sinon la fraction.
    // Le passage a la base 10 se fait en compensant 2^(k-a) et 5^(k-b) : jamais d'arrondi.
    public string EnChaineDecimale(int decimales = 6)
    {
        BigInteger reste = Denominateur;
        int a = 0, b = 0;
        while (reste % 2 == 0) { reste /= 2; a++; }
        while (reste % 5 == 0) { reste /= 5; b++; }
        if (reste != BigInteger.One) return ToString();
        int k = Math.Max(a, b);
        if (k > decimales) return ToString();

        BigInteger num = Numerateur * BigInteger.Pow(2, k - a) * BigInteger.Pow(5, k - b);
        bool negatif = num.Sign < 0;
        if (negatif) num = -num;
        BigInteger echelle = BigInteger.Pow(10, k);
        BigInteger partieEntiere = num / echelle;
        BigInteger resteDecimal = num % echelle;

        string s = partieEntiere.ToString(CultureInfo.InvariantCulture);
        if (k > 0)
        {
            s += "." + resteDecimal.ToString(CultureInfo.InvariantCulture).PadLeft(k, '0');
            if (decimales > k) s += new string('0', decimales - k);
        }
        else if (decimales > 0)
        {
            s += "." + new string('0', decimales);
        }
        return (negatif ? "-" : "") + s;
    }

    public override string ToString() => Denominateur == BigInteger.One
        ? Numerateur.ToString(CultureInfo.InvariantCulture)
        : Numerateur.ToString(CultureInfo.InvariantCulture) + "/" + Denominateur.ToString(CultureInfo.InvariantCulture);
}

public readonly struct CplxC02 : IEquatable<CplxC02>
{
    public RationnelC02 Re { get; }
    public RationnelC02 Im { get; }

    public CplxC02(RationnelC02 re) : this(re, RationnelC02.Zero) { }
    public CplxC02(RationnelC02 re, RationnelC02 im) { Re = re; Im = im; }

    public static readonly CplxC02 Zero = new CplxC02(RationnelC02.Zero, RationnelC02.Zero);
    public static readonly CplxC02 Un = new CplxC02(RationnelC02.Un, RationnelC02.Zero);

    public static CplxC02 operator +(CplxC02 a, CplxC02 b) => new(a.Re + b.Re, a.Im + b.Im);
    public static CplxC02 operator -(CplxC02 a, CplxC02 b) => new(a.Re - b.Re, a.Im - b.Im);
    public static CplxC02 operator *(CplxC02 a, CplxC02 b) =>
        new(a.Re * b.Re - a.Im * b.Im, a.Re * b.Im + a.Im * b.Re);
    public static CplxC02 operator *(CplxC02 a, RationnelC02 k) => new(a.Re * k, a.Im * k);
    public static CplxC02 operator *(RationnelC02 k, CplxC02 a) => a * k;
    public static CplxC02 operator /(CplxC02 a, RationnelC02 k) => new(a.Re / k, a.Im / k);

    public static bool operator ==(CplxC02 a, CplxC02 b) => a.Re == b.Re && a.Im == b.Im;
    public static bool operator !=(CplxC02 a, CplxC02 b) => !(a == b);

    public bool Equals(CplxC02 autre) => this == autre;
    public override bool Equals(object? obj) => obj is CplxC02 c && Equals(c);
    public override int GetHashCode() => HashCode.Combine(Re, Im);

    // |z|^2 exact : jamais une racine carree en double.
    public RationnelC02 ModuleCarre => Re * Re + Im * Im;

    public override string ToString()
    {
        if (Im.Numerateur.Sign == 0) return Re.ToString();
        string signe = Im.Numerateur.Sign < 0 ? " - " : " + ";
        RationnelC02 absolu = Im.Numerateur.Sign < 0 ? -Im : Im;
        return Re.ToString() + signe + absolu.ToString() + "i";
    }
}

public sealed class MatCplxC02
{
    public int Lignes { get; }
    public int Colonnes { get; }
    private readonly CplxC02[][] _e;

    public MatCplxC02(int lignes, int colonnes)
    {
        Lignes = lignes;
        Colonnes = colonnes;
        _e = new CplxC02[lignes][];
        for (int i = 0; i < lignes; i++)
        {
            _e[i] = new CplxC02[colonnes];
            for (int j = 0; j < colonnes; j++) _e[i][j] = CplxC02.Zero;
        }
    }

    public CplxC02 this[int i, int j]
    {
        get => _e[i][j];
        set => _e[i][j] = value;
    }

    public static MatCplxC02 Identite(int n)
    {
        var m = new MatCplxC02(n, n);
        for (int i = 0; i < n; i++) m[i, i] = CplxC02.Un;
        return m;
    }

    public static MatCplxC02 Diagonale(params BigInteger[] valeurs)
    {
        var m = new MatCplxC02(valeurs.Length, valeurs.Length);
        for (int i = 0; i < valeurs.Length; i++) m[i, i] = new CplxC02(new RationnelC02(valeurs[i]));
        return m;
    }

    public static MatCplxC02 Produit(MatCplxC02 a, MatCplxC02 b)
    {
        var m = new MatCplxC02(a.Lignes, b.Colonnes);
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < b.Colonnes; j++)
            {
                CplxC02 somme = CplxC02.Zero;
                for (int k = 0; k < a.Colonnes; k++) somme += a[i, k] * b[k, j];
                m[i, j] = somme;
            }
        return m;
    }

    public static MatCplxC02 Difference(MatCplxC02 a, MatCplxC02 b)
    {
        var m = new MatCplxC02(a.Lignes, a.Colonnes);
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < a.Colonnes; j++) m[i, j] = a[i, j] - b[i, j];
        return m;
    }

    public static MatCplxC02 MultipliePar(MatCplxC02 a, CplxC02 k)
    {
        var m = new MatCplxC02(a.Lignes, a.Colonnes);
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < a.Colonnes; j++) m[i, j] = a[i, j] * k;
        return m;
    }

    // DIVISION VRAIE par un rationnel : a / k. Pour multiplier par 1/2, on divise par 2.
    public static MatCplxC02 DivisePar(MatCplxC02 a, RationnelC02 k)
    {
        var m = new MatCplxC02(a.Lignes, a.Colonnes);
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < a.Colonnes; j++) m[i, j] = a[i, j] / k;
        return m;
    }

    public MatCplxC02 Adjointe()
    {
        var m = new MatCplxC02(Colonnes, Lignes);
        for (int i = 0; i < Lignes; i++)
            for (int j = 0; j < Colonnes; j++)
                m[j, i] = new CplxC02(this[i, j].Re, -this[i, j].Im);
        return m;
    }

    public CplxC02 Trace()
    {
        CplxC02 somme = CplxC02.Zero;
        for (int i = 0; i < Math.Min(Lignes, Colonnes); i++) somme += this[i, i];
        return somme;
    }

    public CplxC02 Determinant2x2()
    {
        if (Lignes != 2 || Colonnes != 2)
            throw new InvalidOperationException("Determinant2x2 : matrice non carree 2x2.");
        return this[0, 0] * this[1, 1] - this[0, 1] * this[1, 0];
    }

    public static bool Egale(MatCplxC02 a, MatCplxC02 b)
    {
        if (a.Lignes != b.Lignes || a.Colonnes != b.Colonnes) return false;
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < a.Colonnes; j++)
                if (a[i, j] != b[i, j]) return false;
        return true;
    }

    public string DiagonaleEnChaine()
    {
        var morceaux = new List<string>();
        for (int i = 0; i < Math.Min(Lignes, Colonnes); i++) morceaux.Add(this[i, i].ToString());
        return "[" + string.Join(", ", morceaux) + "]";
    }

    public string EnChaine()
    {
        var lignes = new List<string>();
        for (int i = 0; i < Lignes; i++)
        {
            var morceaux = new List<string>();
            for (int j = 0; j < Colonnes; j++) morceaux.Add(this[i, j].ToString());
            lignes.Add("[" + string.Join(", ", morceaux) + "]");
        }
        return string.Join("\n", lignes);
    }
}

// Un ket de paire : coefficients (1/sqrt(D)) * P. L'etat reduit vaut (1/D) P P^dagger,
// donc tout reste rationnel exact : la racine carree ne sort jamais du calcul.
public sealed class EtatPaireC02
{
    public int DenominateurCarre { get; }
    public MatCplxC02 P { get; }

    public EtatPaireC02(int denominateurCarre, MatCplxC02 p)
    {
        DenominateurCarre = denominateurCarre;
        P = p;
    }

    public static EtatPaireC02 PaireMaximale(int d1, int d2)
    {
        int m = Math.Min(d1, d2);
        var p = new MatCplxC02(d1, d2);
        for (int i = 0; i < m; i++) p[i, i] = CplxC02.Un;
        return new EtatPaireC02(m, p);
    }

    // L'etat reduit vaut (1/D) P P^dagger : on DIVISE donc par D (entier), pas par 1/D.
    private RationnelC02 Diviseur => new RationnelC02(new BigInteger(DenominateurCarre));

    public MatCplxC02 EtatReduitGauche() =>
        MatCplxC02.DivisePar(MatCplxC02.Produit(P, P.Adjointe()), Diviseur);

    public MatCplxC02 EtatReduitDroit() =>
        MatCplxC02.DivisePar(MatCplxC02.Produit(P.Adjointe(), P), Diviseur);

    public MatCplxC02 CoeurDePaire() => MatCplxC02.DivisePar(MatCplxC02.Produit(P, P), Diviseur);
}

public static class VecteursC02
{
    // (U (x) V) applique a un ket de deux qudits : resultat[k,l] = somme_{i,j} U[k,i] V[l,j] psi[i,j].
    public static CplxC02[] TenseurApplique(MatCplxC02 u, MatCplxC02 v, CplxC02[] psi, int dim)
    {
        var resultat = new CplxC02[dim * dim];
        for (int k = 0; k < dim; k++)
            for (int l = 0; l < dim; l++)
            {
                CplxC02 somme = CplxC02.Zero;
                for (int i = 0; i < dim; i++)
                    for (int j = 0; j < dim; j++)
                        somme += u[k, i] * v[l, j] * psi[(i * dim) + j];
                resultat[(k * dim) + l] = somme;
            }
        return resultat;
    }

    public static bool Egal(CplxC02[] a, CplxC02[] b)
    {
        if (a.Length != b.Length) return false;
        for (int i = 0; i < a.Length; i++) if (a[i] != b[i]) return false;
        return true;
    }

    public static string EnChaine(CplxC02[] v)
    {
        var morceaux = new List<string>();
        foreach (var z in v) morceaux.Add(z.ToString());
        return "[" + string.Join(", ", morceaux) + "]";
    }
}

public static class UnitairesC02
{
    // Unitaire exacte a coefficients rationnels : U = (1/h) [[p, i q],[i q, p]],
    // avec (p, q, h) un triplet pythagoricien, donc p^2 + q^2 = h^2 exactement.
    public static MatCplxC02 Exacte(BigInteger p, BigInteger q, BigInteger h)
    {
        VerifC02.Egale(p * p + q * q == h * h, $"triplet pythagoricien attendu : {p}^2 + {q}^2 = {h}^2");
        var u = new MatCplxC02(2, 2);
        u[0, 0] = new CplxC02(new RationnelC02(p, h));
        u[0, 1] = new CplxC02(RationnelC02.Zero, new RationnelC02(q, h));
        u[1, 0] = new CplxC02(RationnelC02.Zero, new RationnelC02(q, h));
        u[1, 1] = new CplxC02(new RationnelC02(p, h));
        return u;
    }

    public static bool EstUnitaire(MatCplxC02 u) =>
        MatCplxC02.Egale(MatCplxC02.Produit(u, u.Adjointe()), MatCplxC02.Identite(2));
}

// Element de l'anneau Z[w] : a + b*w, avec w = e^{2 pi i / 3} et w^2 = -1 - w.
public readonly struct CycloC02 : IEquatable<CycloC02>
{
    public RationnelC02 A { get; }
    public RationnelC02 B { get; }

    public CycloC02(RationnelC02 a) : this(a, RationnelC02.Zero) { }
    public CycloC02(RationnelC02 a, RationnelC02 b) { A = a; B = b; }

    public static readonly CycloC02 Zero = new CycloC02(RationnelC02.Zero, RationnelC02.Zero);
    public static readonly CycloC02 Un = new CycloC02(RationnelC02.Un, RationnelC02.Zero);
    public static readonly CycloC02 Omega = new CycloC02(RationnelC02.Zero, RationnelC02.Un);
    public static readonly CycloC02 OmegaCarre = new CycloC02(-RationnelC02.Un, -RationnelC02.Un);

    // (a + b w)(c + d w) = (ac - bd) + (ad + bc - bd) w
    public static CycloC02 operator *(CycloC02 x, CycloC02 y) =>
        new(x.A * y.A - x.B * y.B,
            x.A * y.B + x.B * y.A - x.B * y.B);
    public static CycloC02 operator +(CycloC02 x, CycloC02 y) => new(x.A + y.A, x.B + y.B);
    public static CycloC02 operator -(CycloC02 x, CycloC02 y) => new(x.A - y.A, x.B - y.B);

    public static bool operator ==(CycloC02 x, CycloC02 y) => x.A == y.A && x.B == y.B;
    public static bool operator !=(CycloC02 x, CycloC02 y) => !(x == y);

    public bool Equals(CycloC02 autre) => this == autre;
    public override bool Equals(object? obj) => obj is CycloC02 c && Equals(c);
    public override int GetHashCode() => HashCode.Combine(A, B);

    public bool EstNul => A == RationnelC02.Zero && B == RationnelC02.Zero;

    public override string ToString() => B.Numerateur.Sign == 0
        ? A.ToString()
        : A.ToString() + " + " + B.ToString() + "w";
}

public sealed class MatCycloC02
{
    public int Lignes { get; }
    public int Colonnes { get; }
    private readonly CycloC02[][] _e;

    public MatCycloC02(int lignes, int colonnes)
    {
        Lignes = lignes;
        Colonnes = colonnes;
        _e = new CycloC02[lignes][];
        for (int i = 0; i < lignes; i++)
        {
            _e[i] = new CycloC02[colonnes];
            for (int j = 0; j < colonnes; j++) _e[i][j] = CycloC02.Zero;
        }
    }

    public CycloC02 this[int i, int j]
    {
        get => _e[i][j];
        set => _e[i][j] = value;
    }

    public static MatCycloC02 Produit(MatCycloC02 a, MatCycloC02 b)
    {
        var m = new MatCycloC02(a.Lignes, b.Colonnes);
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < b.Colonnes; j++)
            {
                CycloC02 somme = CycloC02.Zero;
                for (int k = 0; k < a.Colonnes; k++) somme += a[i, k] * b[k, j];
                m[i, j] = somme;
            }
        return m;
    }

    public static MatCycloC02 MultipliePar(MatCycloC02 a, CycloC02 k)
    {
        var m = new MatCycloC02(a.Lignes, a.Colonnes);
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < a.Colonnes; j++) m[i, j] = a[i, j] * k;
        return m;
    }

    public static bool Egale(MatCycloC02 a, MatCycloC02 b)
    {
        if (a.Lignes != b.Lignes || a.Colonnes != b.Colonnes) return false;
        for (int i = 0; i < a.Lignes; i++)
            for (int j = 0; j < a.Colonnes; j++)
                if (!(a[i, j] == b[i, j])) return false;
        return true;
    }
}

public static class WeylC02
{
    // Le shift X : X|j> = |j+1 mod d>.
    public static MatCplxC02 Shift(int d)
    {
        var m = new MatCplxC02(d, d);
        for (int i = 0; i < d; i++) m[(i + 1) % d, i] = CplxC02.Un;
        return m;
    }

    // d = 2 (w = -1, reel) et d = 4 (w = i) : les deux tiennent dans le complexe rationnel.
    public static (MatCplxC02 X, MatCplxC02 Z) Lettres(int d, CplxC02 omega)
    {
        var x = Shift(d);
        var z = new MatCplxC02(d, d);
        CplxC02 puissance = CplxC02.Un;
        for (int j = 0; j < d; j++)
        {
            z[j, j] = puissance;
            puissance *= omega;
        }
        return (x, z);
    }

    public static bool RelationWeyl(MatCplxC02 x, MatCplxC02 z, CplxC02 omega) =>
        MatCplxC02.Egale(MatCplxC02.Produit(z, x),
                         MatCplxC02.MultipliePar(MatCplxC02.Produit(x, z), omega));

    // d = 3 : w = e^{2 pi i / 3} est une racine de w^2 + w + 1, l'anneau Z[w] porte le calcul.
    public static (MatCycloC02 X, MatCycloC02 Z) LettresD3()
    {
        var x = new MatCycloC02(3, 3);
        for (int i = 0; i < 3; i++) x[(i + 1) % 3, i] = CycloC02.Un;
        var z = new MatCycloC02(3, 3);
        z[0, 0] = CycloC02.Un;
        z[1, 1] = CycloC02.Omega;
        z[2, 2] = CycloC02.OmegaCarre;
        return (x, z);
    }

    public static bool RelationWeylD3(MatCycloC02 x, MatCycloC02 z) =>
        MatCycloC02.Egale(MatCycloC02.Produit(z, x),
                          MatCycloC02.MultipliePar(MatCycloC02.Produit(x, z), CycloC02.Omega));

    // Deux lettres (a,b) et (c,e) commutent exactement quand w^(b*c - a*e) = 1,
    // c'est-a-dire quand d divise b*c - a*e : le test est entier, donc exact.
    public static bool Commutent(int d, (int a, int b) l1, (int a, int b) l2)
    {
        int k = (l1.b * l2.a) - (l1.a * l2.b);
        return ((k % d) + d) % d == 0;
    }

    public static List<(int a, int b)> LettresListe(int d)
    {
        var liste = new List<(int, int)>();
        for (int a = 0; a < d; a++)
            for (int b = 0; b < d; b++)
                if (a != 0 || b != 0) liste.Add((a, b));
        return liste;
    }

    public static int RecenseNonCommutantes(int d)
    {
        var lettres = LettresListe(d);
        int compte = 0;
        for (int i = 0; i < lettres.Count; i++)
            for (int j = i + 1; j < lettres.Count; j++)
                if (!Commutent(d, lettres[i], lettres[j])) compte++;
        return compte;
    }

    // Cliques maximales du graphe de commutation, par recherche exhaustive.
    public static List<int[]> BasesDuLien(int d)
    {
        var lettres = LettresListe(d);
        int n = lettres.Count;
        var commutent = new bool[n, n];
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++)
                if (i != j) commutent[i, j] = Commutent(d, lettres[i], lettres[j]);

        var cliques = new List<int[]>();
        for (int taille = 1; taille <= d - 1; taille++)
            foreach (var comb in Combinaisons(n, taille))
            {
                bool estClique = true;
                for (int i = 0; i < comb.Length && estClique; i++)
                    for (int j = 0; j < comb.Length; j++)
                        if (i != j && !commutent[comb[i], comb[j]]) { estClique = false; break; }
                if (!estClique) continue;
                bool absorbee = false;
                foreach (var c in cliques)
                    if (EstSousEnsemble(comb, c)) { absorbee = true; break; }
                if (!absorbee) cliques.Add(comb);
            }

        var maximales = new List<int[]>();
        foreach (var c in cliques)
        {
            bool absorbee = false;
            foreach (var c2 in cliques)
                if (EstSousEnsemble(c, c2)) { absorbee = true; break; }
            if (!absorbee) maximales.Add(c);
        }
        return maximales;
    }

    private static bool EstSousEnsemble(int[] petit, int[] grand)
    {
        if (petit.Length >= grand.Length) return false;
        foreach (int x in petit) if (Array.IndexOf(grand, x) < 0) return false;
        return true;
    }

    private static IEnumerable<int[]> Combinaisons(int n, int k)
    {
        var indices = new int[k];
        for (int i = 0; i < k; i++) indices[i] = i;
        while (true)
        {
            var resultat = new int[k];
            Array.Copy(indices, resultat, k);
            yield return resultat;
            int i = k - 1;
            while (i >= 0 && indices[i] == n - k + i) i--;
            if (i < 0) yield break;
            indices[i]++;
            for (int j = i + 1; j < k; j++) indices[j] = indices[j - 1] + 1;
        }
    }
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

.NET 10.0.15 ; arithmetique exacte BigInteger, aucune egalite flottante testee


## 1. Le cadre épistémique — Mesuré, Cité, Ouvert

Ce carnet applique le même patron strict que [Chaînes-01](Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb), décliné sur son objet :

- **Mesuré** : tout ce que le code ci-dessous recalcule de ses propres moyens — les deux sens de l'équivalence du théorème 40.1, le test d'envariance, les valeurs de Born de l'équation 56, l'unitarité du join, la relation de Weyl et les deux recensements (paires de lettres, paires de bases), la borne de Robertson aux sept registres.
- **Cité** : toute la chaîne dérivative de la source — l'axiome et les six définitions, la chaîne des prémices qui court sur douze chapitres au plus long, les comptes du census que nous n'énumérons pas (26 870 appariements, 64 triplets, douze pas, 96 / 2 304 / 84 éléments du groupe d'appui).
- **Ouvert** : ce que la source ne prouve pas — les résultats ouverts 2.4, 2.5, 2.7, 2.8, 18.5, 18.6 sur lesquels les deux théorèmes reposent, le lemme de monogamie pris comme prémice, la borne de valence prise comme entrée, l'échelle du registre laissée ouverte.

**La graduation, préservée telle quelle.** Chaque chapitre du compendium porte une ligne `Standing:`. Pour les chapitres 40 et 41, elle dit exactement : *« Standing: a conditional theorem. »* — un **théorème conditionnel**, conditionnel sur le texte de l'axiome et sur les résultats des chapitres nommés comme prémices, chacun à son propre rang. À titre de contraste, dans la même release, les lois de conservation et le groupe de jauge $SU(3) \times SU(2) \times U(1)$ sont des **théorèmes** (non conditionnels), et 198 questions restent des **résultats ouverts**. Ce carnet ne déplace aucune ligne de cette graduation.

## 2. La paire, sa matrice de coefficients, l'état réduit

L'axiome de Chain Theory ne met au monde que des paires finies de qudits maximalement intriqués. Un ket de paire sur un lien $(d_1, d_2)$ vit dans $\mathbb{C}^{d_1} \otimes \mathbb{C}^{d_2}$ ; son coefficient $C_{ij}$ devant $|i\rangle \otimes |j\rangle$ organise l'état en une **matrice de coefficients** $d_1 \times d_2$. La trace partielle sur le bout lointain donne l'état réduit au bout proche : $\rho_{\text{proche}} = C C^\dagger$ (Cité ; c'est l'éq. 51 du chapitre 40, reproduite ci-dessous au théorème 40.1).

Le ket de paire maximalement intriqué de Chaînes-01, $|\omega_m\rangle$ sur le support commun $m = \min(d_1, d_2)$, a pour matrice de coefficients la troncature diagonale $C_{ii} = 1/\sqrt{m}$ — à un facteur près, l'identité plongée : une isométrie partielle.

In [2]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var C22 = EtatPaireC02.PaireMaximale(2, 2);
var rhoGauche = C22.EtatReduitGauche();
var rhoDroit = C22.EtatReduitDroit();
var identiteDemi = MatCplxC02.DivisePar(MatCplxC02.Identite(2), new RationnelC02(new BigInteger(2)));

Console.WriteLine("rho_gauche =");
Console.WriteLine(rhoGauche.EnChaine());
Console.WriteLine("rho_gauche == I/2 : " + MatCplxC02.Egale(rhoGauche, identiteDemi));
Console.WriteLine("rho_droit  == I/2 : " + MatCplxC02.Egale(rhoDroit, identiteDemi));
VerifC02.Egale(MatCplxC02.Egale(rhoGauche, identiteDemi), "rho_gauche doit valoir I/2 sur le lien (2,2)");
VerifC02.Egale(MatCplxC02.Egale(rhoDroit, identiteDemi), "rho_droit doit valoir I/2 sur le lien (2,2)");

// Le lien (3,2) : support commun de rang 2, etat reduit plat SUR LE SUPPORT
var C32 = EtatPaireC02.PaireMaximale(3, 2);
var rhoQutrit = C32.EtatReduitGauche();
Console.WriteLine("(3,2) valeurs propres cote qutrit : " + rhoQutrit.DiagonaleEnChaine());
for (int i = 0; i < 3; i++)
    for (int j = 0; j < 3; j++)
        if (i != j)
            VerifC02.Egale(rhoQutrit[i, j] == CplxC02.Zero, $"rho_qutrit doit etre diagonale ({i},{j})");
VerifC02.Egale(rhoQutrit[0, 0].Re == new RationnelC02(BigInteger.One, new BigInteger(2)), "rho_qutrit[0,0] doit valoir 1/2");
VerifC02.Egale(rhoQutrit[1, 1].Re == new RationnelC02(BigInteger.One, new BigInteger(2)), "rho_qutrit[1,1] doit valoir 1/2");
VerifC02.Egale(rhoQutrit[2, 2].Re == RationnelC02.Zero, "rho_qutrit[2,2] doit valoir 0");


rho_gauche =


[1/2, 0]
[0, 1/2]


rho_gauche == I/2 : True


rho_droit  == I/2 : True


(3,2) valeurs propres cote qutrit : [1/2, 1/2, 0]


### Lecture du résultat

Mesuré : sur le lien $(2,2)$, $\rho = CC^\dagger = I/2$ aux deux bouts — plat, en arithmétique exacte
(le facteur $1/\sqrt 2$ du ket de paire est porté symboliquement, jamais approché). Sur le lien $(3,2)$, côté qutrit, les valeurs propres sont $\{1/2, 1/2, 0\}$ : l'état réduit est plat **sur le support de rang 2** et nul hors de lui — exactement la nuance que le théorème suivant énonce.

## 3. Théorème 40.1 — le pli plat, et l'unitarité du transport, du join et du pas

> **Theorem 40.1 (The flat pair and the unitary transport, join and step).** *This result is conditional on the axiom text and on the results of the chapters named as its premises in the derivation below, each at its own standing.* A pure pair on a common support of rank $m$ has a flat reduced state exactly when its coefficient matrix is a multiple of a unitary on the support, and the standing pair state is such a multiple at every alphabet dimension and every pair type, so the transport and the join are unitary as corollaries and the order step is unitary by its central generator: the read transport is a scaled partial isometry at all 26 870 tabulated matchings, the join at a shared qudit a unitary at all 64 triples, and the order step a unitary at the twelve steps. — *Standing: a conditional theorem.*

La partie **exacte** tient en une ligne (éq. 51) :

$$\rho_{\text{proche}} = CC^\dagger = \lambda P_m, \qquad \lambda = 1 \ \text{(au ket de paire non normalisé)}, \quad \operatorname{rang} P_m = m.$$

État réduit plat sur le support $\iff$ $C$ est multiple d'une unitaire sur ce support. La **correspondance nommée** avec la littérature : c'est le lemme de Schmidt (une état biparti pur a un état réduit maximalement mixte sur un support de rang $m$ si et seulement si sa matrice de coefficients est multiple d'une unitaire sur ce support, réf. [33] de la source) ; l'évolution d'un système fermé est unitaire. Les comptes du census — 26 870 appariements, 64 triplets, douze pas — sont **Cités** (machinerie du compendium, non rejouée ici) ; les deux sens de l'équivalence, eux, se rejouent en trois lignes.

In [3]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var U = UnitairesC02.Exacte(3, 4, 5);
VerifC02.Egale(UnitairesC02.EstUnitaire(U), "U = [[3/5, 4i/5],[4i/5, 3/5]] doit etre unitaire");

// Sens (i) : C multiple d'une unitaire -> etat reduit plat
var Cu = new EtatPaireC02(2, U);                 // C = U / sqrt(2)
var rhoU = Cu.EtatReduitGauche();
Console.WriteLine("(i) C = U/sqrt(2) : valeurs propres de CC^dagger : " + rhoU.DiagonaleEnChaine());
VerifC02.Egale(MatCplxC02.Egale(rhoU, identiteDemi), "C = U/sqrt(2) doit donner l'etat plat I/2");

// Sens (ii) : C = U diag(1,2)/sqrt(5) : ni multiple d'une unitaire, etat reduit non plat
var Pnp = MatCplxC02.Produit(U, MatCplxC02.Diagonale(1, 2));
var Cnp = new EtatPaireC02(5, Pnp);              // C = U diag(1,2) / sqrt(5)
var rhoNp = Cnp.EtatReduitGauche();               // (1/5) P P^dagger
var ctcNp = Cnp.EtatReduitDroit();                // (1/5) P^dagger P : valeurs singulieres au carre
var quart = new RationnelC02(BigInteger.One, new BigInteger(4));
var vingtCinq = new RationnelC02(BigInteger.One, new BigInteger(25));
Console.WriteLine("(ii) C = U diag(1,2)/sqrt(5) : trace de CC^dagger = " + rhoNp.Trace().Re.EnChaineDecimale()
    + " , determinant = " + rhoNp.Determinant2x2().Re.EnChaineDecimale());
Console.WriteLine("    valeurs propres de CC^dagger : 1/5 et 4/5 (polynome t^2 - t + 4/25) -> non plat");
Console.WriteLine("    valeurs singulieres au carre : 1/5 et 4/5 (la source imprime 2/sqrt(5) et 1/sqrt(5) -- CITE, hors arithmetique rationnelle)");
VerifC02.Egale(rhoNp.Trace().Re == RationnelC02.Un, "trace de CC^dagger : 1 attendue");
VerifC02.Egale(rhoNp.Determinant2x2().Re == new RationnelC02(new BigInteger(4), new BigInteger(25)), "determinant : 4/25 attendu");
VerifC02.Egale(ctcNp.Trace().Re == RationnelC02.Un, "trace de C^dagger C : 1 attendue");
VerifC02.Egale(ctcNp.Determinant2x2().Re == new RationnelC02(new BigInteger(4), new BigInteger(25)), "determinant de C^dagger C : 4/25 attendu");

// Sur un support propre, les deux sens : le ket de paire (3,2)
var C32bis = EtatPaireC02.PaireMaximale(3, 2);
var ctc32 = C32bis.EtatReduitDroit();
Console.WriteLine("(3,2) : C^dagger C == I/2 : " + MatCplxC02.Egale(ctc32, identiteDemi)
    + " (multiple de l'unite sur le support de rang 2)");
VerifC02.Egale(MatCplxC02.Egale(ctc32, identiteDemi), "C^dagger C doit valoir I/2 sur le support de rang 2");


(i) C = U/sqrt(2) : valeurs propres de CC^dagger : [1/2, 1/2]


(ii) C = U diag(1,2)/sqrt(5) : trace de CC^dagger = 1.000000 , determinant = 0.160000


    valeurs propres de CC^dagger : 1/5 et 4/5 (polynome t^2 - t + 4/25) -> non plat


    valeurs singulieres au carre : 1/5 et 4/5 (la source imprime 2/sqrt(5) et 1/sqrt(5) -- CITE, hors arithmetique rationnelle)


(3,2) : C^dagger C == I/2 : True (multiple de l'unite sur le support de rang 2)


### Lecture du résultat

Mesuré : les deux sens de l'équivalence tiennent. Une matrice $C$ multiple d'une unitaire (au facteur $1/\sqrt 2$ près) donne un état réduit aux valeurs propres égales ; une matrice $C$ qui n'en est pas multiple — $U\,\mathrm{diag}(1,2)/\sqrt 5$ — a des valeurs propres **exactes** $1/5$ et $4/5$ (trace $1$, déterminant $4/25$) et un état réduit non
plat ; les valeurs singulières $2/\sqrt 5$ et $1/\sqrt 5$ de la source restent **citées** — elles ne sont pas
rationnelles et ce carnet ne les approxime pas. Sur le lien $(3,2)$, la matrice rectangulaire satisfait $C^\dagger C = I/2$ : **multiple de l'unitaire sur le support** de rang 2, isométrie partielle au-delà. C'est l'éq. 51, dans les deux sens, sur nos propres matrices.

## 4. Le groupe d'appariement et l'envariance

La dérivation de la règle de Born du chapitre 40 s'appuie sur le **groupe d'appariement** : au ket de paire maximalement intriqué, toute unitaire sur un bout est défait par l'unitaire conjuguée sur l'autre bout — $(U \otimes \bar U)|\omega\rangle = |\omega\rangle$. La correspondance nommée : c'est l'**envariance** de Zurek, l'invariance d'un état intriqué sous une unitaire sur le système qu'une unitaire sur l'environnement défait (réfs. [71, 72] de la source) ; à une paire maximalement intriquée, *toute* unitaire est défaite, et ces paires d'unitaires forment le groupe d'appariement du chapitre. Le texte précise : ces paires de lecture commutent entre sites disjoints et le relevé multi-sites est le produit des relevés mono-site (Cité — le lemme de monogamie y est pris comme prémice).

In [4]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var omegaKet = new CplxC02[] { CplxC02.Un, CplxC02.Zero, CplxC02.Zero, CplxC02.Un };   // |00> + |11>, non normalise
var couples = new List<BigInteger[]>
{
    new BigInteger[] { 3, 4, 5 }, new BigInteger[] { 5, 12, 13 }, new BigInteger[] { 8, 15, 17 },
    new BigInteger[] { 7, 24, 25 }, new BigInteger[] { 20, 21, 29 }
};

bool invariant = true;
foreach (var pq in couples)
{
    var Q = UnitairesC02.Exacte(pq[0], pq[1], pq[2]);
    invariant &= VecteursC02.Egal(VecteursC02.TenseurApplique(Q, Q.Adjointe(), omegaKet, 2), omegaKet);
}
Console.WriteLine("5 unitaires exactes : (U (x) Ubar)|omega> == |omega> : " + invariant);
VerifC02.Egale(invariant, "l'envariance doit tenir pour les cinq unitaires exactes");

bool deplace = false;
foreach (var pq in couples)
{
    var Q = UnitairesC02.Exacte(pq[0], pq[1], pq[2]);
    deplace |= !VecteursC02.Egal(VecteursC02.TenseurApplique(Q, Q, omegaKet, 2), omegaKet);
}
Console.WriteLine("5 paires quelconques (U (x) V)|omega> != |omega> : " + deplace);
VerifC02.Egale(deplace, "sans conjugaison au second facteur, l'etat doit etre deplace");


5 unitaires exactes : (U (x) Ubar)|omega> == |omega> : True


5 paires quelconques (U (x) V)|omega> != |omega> : True


### Lecture du résultat

Mesuré : pour cinq unitaires aléatoires, $(U \otimes \bar U)|\omega\rangle = |\omega\rangle$ à l'**égalité exacte** — l'état de paire est un point fixe du groupe d'appariement tout entier ; et cinq paires quelconques $(U \otimes V)$ sans conjugaison le déplacent toutes. L'envariance n'est pas une propriété génique des unitaires produit : elle est portée par la **conjugaison**, c'est-à-dire par la structure de paire elle-même.

## 5. La règle de Born : le canon $\mathrm{Tr}(\rho P)$

Le chapitre 40 dérive la règle de probabilité du « lavage » : le relevé debout est la probabilité de l'issue, *par théorème, conditionnel sur le texte de l'axiome et sur deux importations de son résultat parent* (l'espace de Hilbert d'un qudit, la paire comme rayon). Le canon : **la trace du projecteur contre l'état, sur la dimension du registre** — toute quantité connue est un fonctionnel invariant sous le groupe d'appariement, la trace normalisée sur le registre est l'unique état invariant, donc ce canon est l'unique relevé (Cité). La correspondance nommée : la règle de Born de la littérature assigne à l'issue de projecteur $P$ sur l'état $\rho$ la probabilité $\mathrm{Tr}(\rho P)$ — seule fonction-cadre de son espèce en dimension $\geq 3$ par le théorème de Gleason (réf. [70]).

Le plus petit exemple est chiffré à la source (éq. 56) aux observables d'étiquette du §8 ci-dessous : au registre plein, $\langle A \rangle = \langle B \rangle = 1/2$.

In [5]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var A = new MatCplxC02(2, 2);
A[1, 1] = CplxC02.Un;                                                       // projecteur d'etiquette 1, base Z
var demi = new RationnelC02(BigInteger.One, new BigInteger(2));
var B = new MatCplxC02(2, 2);                                               // projecteur d'etiquette 1, base X
B[0, 0] = new CplxC02(demi); B[0, 1] = new CplxC02(demi);
B[1, 0] = new CplxC02(demi); B[1, 1] = new CplxC02(demi);
var rhoPlein = MatCplxC02.DivisePar(MatCplxC02.Identite(2), new RationnelC02(new BigInteger(2)));
Console.WriteLine("<A>_plein = " + MatCplxC02.Produit(rhoPlein, A).Trace().Re.EnChaineDecimale());
Console.WriteLine("<B>_plein = " + MatCplxC02.Produit(rhoPlein, B).Trace().Re.EnChaineDecimale());
VerifC02.Egale(MatCplxC02.Produit(rhoPlein, A).Trace().Re == demi, "<A>_plein doit valoir 1/2");
VerifC02.Egale(MatCplxC02.Produit(rhoPlein, B).Trace().Re == demi, "<B>_plein doit valoir 1/2");

// Normalisation : sur chaque base, la somme des sorties de Born fait 1
var baseZ = new List<MatCplxC02> { MatCplxC02.Diagonale(1, 0), MatCplxC02.Diagonale(0, 1) };
var projecteurX = new MatCplxC02(2, 2);
projecteurX[0, 0] = new CplxC02(demi);
projecteurX[0, 1] = new CplxC02(-demi);
projecteurX[1, 0] = new CplxC02(-demi);
projecteurX[1, 1] = new CplxC02(demi);
var baseX = new List<MatCplxC02> { projecteurX, B };
RationnelC02 sommeZ = RationnelC02.Zero, sommeX = RationnelC02.Zero;
foreach (var p in baseZ) sommeZ += MatCplxC02.Produit(rhoPlein, p).Trace().Re;
foreach (var p in baseX) sommeX += MatCplxC02.Produit(rhoPlein, p).Trace().Re;
Console.WriteLine("somme Born base Z = " + sommeZ.EnChaineDecimale());
Console.WriteLine("somme Born base X = " + sommeX.EnChaineDecimale());
VerifC02.Egale(sommeZ == RationnelC02.Un, "la somme des Born sur la base Z doit valoir 1");
VerifC02.Egale(sommeX == RationnelC02.Un, "la somme des Born sur la base X doit valoir 1");


<A>_plein = 0.500000


<B>_plein = 0.500000


somme Born base Z = 1.000000


somme Born base X = 1.000000


### Lecture du résultat

Mesuré : $\langle A \rangle_{\text{plein}} = \langle B \rangle_{\text{plein}} = 0{,}5$ — l'équation 56 de la source, recalculée depuis nos propres matrices. Et le canon se normalise : sur chacune des deux bases, la somme des $\mathrm{Tr}(\rho P)$ sur les projecteurs d'étiquette vaut 1. C'est tout ce que le mot « probabilité » exige — obtenu ici comme l'unique relevé invariant, pas posé comme postulat.

## 6. Le join de la figure 15 — une unitaire sur son support

Le théorème 40.1 fait du **join** une conséquence : la contraction de deux kets de paire sur un site partagé est unitaire sur le support composé. La figure 15 de la source montre le plus petit join : deux kets de paire sur des liens $(2,2)$ partageant le qubit du site du milieu, chaque site étiqueté par sa dimension. Le census de la source vérifie le join **à tous les 64 triplets** (Cité) ; au plus petit triplet, la contraction se rejoue comme un simple produit de matrices : la matrice du premier lien ($2 \times 2$, de $A$ vers $B$) fois celle du second ($2 \times 2$, de $B$ vers $C$).

In [6]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var jDiag = C22.CoeurDePaire();
Console.WriteLine("join diagonal : J =");
Console.WriteLine(jDiag.EnChaine());
var jjDiag = MatCplxC02.Produit(jDiag, jDiag.Adjointe());
Console.WriteLine("J J^dagger =");
Console.WriteLine(jjDiag.EnChaine());
var quartIdentite = MatCplxC02.DivisePar(MatCplxC02.Identite(2), new RationnelC02(new BigInteger(4)));
VerifC02.Egale(MatCplxC02.Egale(jjDiag, quartIdentite), "J J^dagger doit valoir I/4");

// Avec des bases tournees aux extremites : le join reste unitaire a un facteur pres
var U1 = UnitairesC02.Exacte(3, 4, 5);
var U2 = UnitairesC02.Exacte(5, 12, 13);
var jRot = MatCplxC02.DivisePar(MatCplxC02.Produit(U1, U2), new RationnelC02(new BigInteger(2)));
var jRotAdjoint = MatCplxC02.Produit(jRot.Adjointe(), jRot);
Console.WriteLine("join de deux paires tournees : J^dagger J =");
Console.WriteLine(jRotAdjoint.EnChaine());
VerifC02.Egale(MatCplxC02.Egale(jRotAdjoint, quartIdentite), "J^dagger J doit valoir I/4");


join diagonal : J =


[1/2, 0]
[0, 1/2]


J J^dagger =


[1/4, 0]
[0, 1/4]


join de deux paires tournees : J^dagger J =


[1/4, 0]
[0, 1/4]


### Lecture du résultat

Mesuré : le join diagonal $J = J J^\dagger$ vaut $I/2$, donc $J J^\dagger = \tfrac14 I$ ; et le join de deux paires aux bases tournées satisfait $J^\dagger J = \tfrac14 I$ — chaque facteur est une unitaire au facteur $1/\sqrt2$ près, donc le produit est une unitaire au facteur $1/2$ près, et $J^\dagger J = \tfrac14 I$ (c'est bien $I/4$ que la cellule affiche, pas $I$). La contraction sur le site partagé ne fabrique jamais une non-unitarité : c'est le corollaire du théorème 40.1, vu ici sur le plus petit triplet de la figure 15.

## 7. Les lettres de Weyl aux dimensions 2, 3 et 4

Le chapitre 41 tire la relation d'incertitude de la **relation de Weyl**. Les lettres de Weyl en dimension $d$ : le *shift* $X|j\rangle = |j+1 \bmod d\rangle$ et le *clock* $Z|j\rangle = \omega^j |j\rangle$, avec $\omega = e^{2i\pi/d}$ — la correspondance nommée avec la relation de Weyl de la littérature (réf. [23]). Le théorème vérifie la relation « à chaque paire de lettres aux dimensions deux, trois et quatre », et le census compte les paires de lettres au commutateur non nul : **3, 24 et 84** (Cité ; nous les recomptons ci-dessous). La convention est épinglée : $ZX = \omega \cdot XZ$, et deux lettres commutent exactement quand leur phase de Weyl vaut 1.

In [7]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var omegaD2 = new CplxC02(new RationnelC02(-BigInteger.One));   // d = 2 : w = -1
var omegaD4 = new CplxC02(RationnelC02.Zero, RationnelC02.Un);   // d = 4 : w = i
var (x2, z2) = WeylC02.Lettres(2, omegaD2);
Console.WriteLine("d=2 : ZX == w * XZ : " + WeylC02.RelationWeyl(x2, z2, omegaD2));
var (x3, z3) = WeylC02.LettresD3();
Console.WriteLine("d=3 : ZX == w * XZ : " + WeylC02.RelationWeylD3(x3, z3));
var (x4, z4) = WeylC02.Lettres(4, omegaD4);
Console.WriteLine("d=4 : ZX == w * XZ : " + WeylC02.RelationWeyl(x4, z4, omegaD4));
VerifC02.Egale(WeylC02.RelationWeyl(x2, z2, omegaD2), "relation de Weyl en d=2");
VerifC02.Egale(WeylC02.RelationWeylD3(x3, z3), "relation de Weyl en d=3");
VerifC02.Egale(WeylC02.RelationWeyl(x4, z4, omegaD4), "relation de Weyl en d=4");

// Census de la source (donnee citee, chapitre 41) : nombre de paires de lettres
// au commutateur non nul aux dimensions 2, 3 et 4.
var censusLettres = new Dictionary<int, int> { { 2, 3 }, { 3, 24 }, { 4, 84 } };
Console.WriteLine();
var conformeLettres = new List<bool>();
foreach (int d in new[] { 2, 3, 4 })
{
    var lettres = WeylC02.LettresListe(d);
    int total = lettres.Count * (lettres.Count - 1) / 2;
    int nonNul = WeylC02.RecenseNonCommutantes(d);
    int census = censusLettres[d];
    bool ok = nonNul == census;
    conformeLettres.Add(ok);
    Console.WriteLine($"d={d} : {lettres.Count} lettres, {total} paires, {nonNul} non commutantes"
        + $" | census source : {census} | conforme : {ok}");
    VerifC02.Egale(ok, $"census des paires de lettres en d={d} : {nonNul} mesure contre {census} cite");
}
Console.WriteLine("conformite au census des paires de lettres (d=2,3,4) : " + conformeLettres.TrueForAll(x => x));


d=2 : ZX == w * XZ : True


d=3 : ZX == w * XZ : True


d=4 : ZX == w * XZ : True


d=2 : 3 lettres, 3 paires, 3 non commutantes | census source : 3 | conforme : True


d=3 : 8 lettres, 28 paires, 24 non commutantes | census source : 24 | conforme : True


d=4 : 15 lettres, 105 paires, 84 non commutantes | census source : 84 | conforme : True


conformite au census des paires de lettres (d=2,3,4) : True


### Lecture du résultat

Mesuré : la relation de Weyl $ZX = \omega \cdot XZ$ tient aux trois dimensions, et notre recensement indépendant des paires de lettres non commutantes rend **3, 24 et 84** — les trois comptes du census de la source, retrouvés par énumération exhaustive des $d^2 - 1$ lettres. En $d=3$, le calcul n'utilise aucun complexe flottant : $\omega = e^{2i\pi/3}$ est une racine de
$w^2 + w + 1$, et la relation est vérifiée dans l'anneau cyclotomique $\mathbb{Z}[\omega]$ (coefficients
rationnels). Le test de commutation de deux lettres est un **test entier** — $d \mid bc - ae$ — donc exact
là où le jumeau Python compare un complexe flottant à $1$.

La conformité n'est pas laissée à la lecture : le code épingle le census de la source comme donnée et compare lui-même, dimension par dimension, le compte mesuré au compte cité — verdicts **conforme : True** aux trois dimensions (3 ↔ 3, 24 ↔ 24, 84 ↔ 84), verdict global **True**. À $d=3$ : 28 paires dont 24 non commutantes, les 4 restantes étant les paires proportionnelles (les 4 droites du plan $\mathbb{F}_3^2$). À $d=4$ : 105 paires dont 84 non commutantes, 21 commutantes.


## 8. Les bases du lien, les observables d'étiquette — l'équation 55

Une base du lien à un bout de dimension $d$ est un **ensemble maximal de lettres de Weyl deux à deux commutantes**, avec ses $d$ projecteurs-joints propres de rang un ; l'**observable d'étiquette** de la base porte l'étiquette de chaque projecteur-joint comme valeur propre, les étiquettes allant de $0$ à $d-1$ (l'étiquetage est une convention du relevé). Au bout qubit, une observable d'étiquette est le projecteur de rang un sur le vecteur propre étiqueté 1.

Au plus petit cas — le bout qubit, bases de la première et de la seconde lettre — la source écrit (éq. 55) :

$$A = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix}, \qquad B = \frac{1}{2}\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}, \qquad [A, B] = \begin{pmatrix} 0 & -\tfrac{1}{2} \\ \tfrac{1}{2} & 0 \end{pmatrix}.$$

Deux lectures du même lien dans deux bases ne commutent pas : le commutateur des deux observables d'étiquette est une **matrice exacte non nulle** à chaque paire de bases distinctes, et nulle d'une base contre elle-même. Le census de la source énumère les paires de bases distinctes aux dimensions 2, 3, 4 : **trois, six et vingt-et-une paires non ordonnées** — nous les recomptons.

In [8]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var commAB = MatCplxC02.Difference(MatCplxC02.Produit(A, B), MatCplxC02.Produit(B, A));
Console.WriteLine("A =");
Console.WriteLine(A.EnChaine());
Console.WriteLine("B =");
Console.WriteLine(B.EnChaine());
Console.WriteLine("[A, B] =");
Console.WriteLine(commAB.EnChaine());
VerifC02.Egale(commAB[0, 1].Re == new RationnelC02(BigInteger.One, new BigInteger(-2)), "[A,B][0,1] doit valoir -1/2");
VerifC02.Egale(commAB[1, 0].Re == new RationnelC02(BigInteger.One, new BigInteger(2)), "[A,B][1,0] doit valoir 1/2");
VerifC02.Egale(commAB[0, 0] == CplxC02.Zero && commAB[1, 1] == CplxC02.Zero, "[A,B] doit etre antihermitienne de trace nulle");

// Census de la source (donnee citee, chapitre 41) : nombre de bases du lien et
// nombre de paires de bases distinctes aux dimensions 2, 3 et 4.
var censusBases = new Dictionary<int, int> { { 2, 3 }, { 3, 4 }, { 4, 7 } };
var censusPairesBases = new Dictionary<int, int> { { 2, 3 }, { 3, 6 }, { 4, 21 } };
Console.WriteLine();
var conformeBases = new List<bool>();
var conformePairesBases = new List<bool>();
foreach (int d in new[] { 2, 3, 4 })
{
    var bases = WeylC02.BasesDuLien(d);
    int paires = bases.Count * (bases.Count - 1) / 2;
    int censusB = censusBases[d];
    int censusP = censusPairesBases[d];
    bool okBases = bases.Count == censusB;
    bool okPaires = paires == censusP;
    conformeBases.Add(okBases);
    conformePairesBases.Add(okPaires);
    Console.WriteLine($"d={d} : {bases.Count} bases, {paires} paires distinctes"
        + $" | census source : {censusB} bases, {censusP} paires"
        + $" | conforme bases : {okBases} | conforme paires : {okPaires}");
    VerifC02.Egale(okBases, $"census des bases en d={d} : {bases.Count} mesure contre {censusB} cite");
    VerifC02.Egale(okPaires, $"census des paires de bases en d={d} : {paires} mesure contre {censusP} cite");
}
Console.WriteLine("conformite au census des bases (d=2,3,4) : " + conformeBases.TrueForAll(x => x));
Console.WriteLine("conformite au census des paires de bases (d=2,3,4) : " + conformePairesBases.TrueForAll(x => x));


A =


[0, 0]
[0, 1]


B =


[1/2, 1/2]
[1/2, 1/2]


[A, B] =


[0, -1/2]
[1/2, 0]


d=2 : 3 bases, 3 paires distinctes | census source : 3 bases, 3 paires | conforme bases : True | conforme paires : True


d=3 : 4 bases, 6 paires distinctes | census source : 4 bases, 6 paires | conforme bases : True | conforme paires : True


d=4 : 7 bases, 21 paires distinctes | census source : 7 bases, 21 paires | conforme bases : True | conforme paires : True


conformite au census des bases (d=2,3,4) : True


conformite au census des paires de bases (d=2,3,4) : True


### Lecture du résultat

Mesuré : l'équation 55 est reproduite au centième près — $A$, $B$ et le commutateur exact $[A,B]$ aux entrées $\pm 1/2$. Et notre énumération indépendante des ensembles maximaux de lettres commutantes rend **3 bases à $d=2$, 4 à $d=3$, 7 à $d=4$**, soit les **3, 6 et 21 paires** du census de la source. Comme pour les lettres, la conformité au census est **dérivée par le code** : le census est épinglé comme donnée, comparé aux comptes mesurés, et le verdict est calculé dimension par dimension — **conforme bases : True** et **conforme paires : True** aux trois dimensions, verdicts globaux **True**. À $d=3$ ce sont les 4 bases mutuellement incongruentes attendues ; à $d=4$ le compte de 7 — et non $d+1 = 5$ — vient de ce que la dimension n'est pas première : les cliques maximales du graphe de commutation ne sont pas toutes des droites.


## 9. La borne de Robertson aux sept registres — équation 57

> **Theorem 41.1 (The uncertainty relation from the Weyl relation).** *This result is conditional on the axiom text and on the results of the chapters named as its premises in the derivation below, each at its own standing.* Two reads of the one link in two bases do not commute: the commutator of the two label observables is a nonzero exact matrix at every pair of distinct bases at the dimensions two, three and four and zero at a basis against itself; the variance of every read at every register of the span equals the norm of the centred read; the product of the two variances is at least the squared modulus of half the commutator expectation at every pair of distinct bases and every register, transported by the whole standing group, with no violating pair and state at any dimension; and the saturating registers are the kernel of the Cauchy-Schwarz defect at every pair, the full register among them at none. — *Standing: a conditional theorem.*

La correspondance nommée : la forme de **Robertson** de la relation d'incertitude, $\Delta A\, \Delta B \geq \tfrac{1}{2}|\langle[A,B]\rangle|$, corollaire de Cauchy-Schwarz avec égalité ssi les opérateurs centrés appliqués à l'état sont proportionnels (réf. [73]). Au plus petit cas, la source écrit l'éq. 57 au registre saturé : $\operatorname{Var}(A)\operatorname{Var}(B) \geq \left(\tfrac{1}{2}\langle[A,B]\rangle\right)^2$, $0 \cdot \tfrac14 \geq 0$. Les **registres du voisinage** au bout qubit : les vecteurs propres des trois bases (six registres purs) plus le registre plein — sept en tout. La source mesure : six d'entre eux saturent, le registre plein non, avec les deux variances à $1/4$, leur produit à $1/16$ et l'écart à $1/16$.

In [9]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

var ketZero = new MatCplxC02(2, 2);
ketZero[0, 0] = CplxC02.Un;
var ketUn = new MatCplxC02(2, 2);
ketUn[1, 1] = CplxC02.Un;
var ketPlus = new MatCplxC02(2, 2);
ketPlus[0, 0] = CplxC02.Un; ketPlus[0, 1] = CplxC02.Un;
ketPlus[1, 0] = CplxC02.Un; ketPlus[1, 1] = CplxC02.Un;
ketPlus = MatCplxC02.DivisePar(ketPlus, new RationnelC02(new BigInteger(2)));
var ketMoins = new MatCplxC02(2, 2);
ketMoins[0, 0] = CplxC02.Un; ketMoins[0, 1] = new CplxC02(-demi);
ketMoins[1, 0] = new CplxC02(-demi); ketMoins[1, 1] = CplxC02.Un;
var ketYPlus = new MatCplxC02(2, 2);
ketYPlus[0, 0] = new CplxC02(demi); ketYPlus[0, 1] = new CplxC02(RationnelC02.Zero, -demi);
ketYPlus[1, 0] = new CplxC02(RationnelC02.Zero, demi); ketYPlus[1, 1] = new CplxC02(demi);
var ketYMoins = new MatCplxC02(2, 2);
ketYMoins[0, 0] = new CplxC02(demi); ketYMoins[0, 1] = new CplxC02(RationnelC02.Zero, demi);
ketYMoins[1, 0] = new CplxC02(RationnelC02.Zero, -demi); ketYMoins[1, 1] = new CplxC02(demi);

var registres = new List<(string Nom, MatCplxC02 Rho)>
{
    ("|0>", ketZero), ("|1>", ketUn), ("|+>", ketPlus), ("|->", ketMoins),
    ("|y+>", ketYPlus), ("|y->", ketYMoins)
};

RationnelC02 LectureVariance(MatCplxC02 rho, MatCplxC02 observable, out RationnelC02 esperance)
{
    esperance = MatCplxC02.Produit(rho, observable).Trace().Re;
    RationnelC02 second = MatCplxC02.Produit(rho, MatCplxC02.Produit(observable, observable)).Trace().Re;
    return second - esperance * esperance;
}

Console.WriteLine($"{"registre",-9} {"Var(A)",-9} {"Var(B)",-9} {"produit",-9} {"borne",-9} sature");
int saturations = 0;
foreach (var (nom, rho) in registres)
{
    RationnelC02 vA = LectureVariance(rho, A, out RationnelC02 eA);
    RationnelC02 vB = LectureVariance(rho, B, out RationnelC02 eB);
    RationnelC02 produit = vA * vB;
    // Borne de Robertson : Var(A) Var(B) >= (1/2 |<[A,B]>|)^2 = |Tr(rho [A,B])|^2 / 4.
    RationnelC02 borne = (MatCplxC02.Produit(rho, commAB).Trace().ModuleCarre) / new RationnelC02(new BigInteger(4));
    bool sature = produit == borne;
    if (sature) saturations++;
    Console.WriteLine($"{nom,-9} {vA.EnChaineDecimale(),-9} {vB.EnChaineDecimale(),-9} {produit.EnChaineDecimale(),-9} {borne.EnChaineDecimale(),-9} {sature}");
}
RationnelC02 vAPlein = LectureVariance(rhoPlein, A, out RationnelC02 eAPlein);
RationnelC02 vBPlein = LectureVariance(rhoPlein, B, out RationnelC02 eBPlein);
RationnelC02 produitPlein = vAPlein * vBPlein;
RationnelC02 bornePlein = (MatCplxC02.Produit(rhoPlein, commAB).Trace().ModuleCarre) / new RationnelC02(new BigInteger(4));
bool saturePlein = produitPlein == bornePlein;
Console.WriteLine($"{"plein",-9} {vAPlein.EnChaineDecimale(),-9} {vBPlein.EnChaineDecimale(),-9} {produitPlein.EnChaineDecimale(),-9} {bornePlein.EnChaineDecimale(),-9} {saturePlein}");
Console.WriteLine("  produit = 1/16 : " + (produitPlein == new RationnelC02(BigInteger.One, new BigInteger(16)))
    + " ; ecart a la borne = 1/16 : " + ((produitPlein - bornePlein) == new RationnelC02(BigInteger.One, new BigInteger(16))));
Console.WriteLine($"\nsaturations : {saturations} registres purs sur 6 ; source : six sur sept, le plein exclus");
VerifC02.Egale(saturations == 6, "six registres purs doivent saturer");
VerifC02.Egale(!saturePlein, "le registre plein ne doit pas saturer");
VerifC02.Egale(produitPlein == new RationnelC02(BigInteger.One, new BigInteger(16)), "produit du registre plein : 1/16");
VerifC02.Egale((produitPlein - bornePlein) == new RationnelC02(BigInteger.One, new BigInteger(16)), "ecart du registre plein : 1/16");


registre  Var(A)    Var(B)    produit   borne     sature


|0>       0.000000  0.250000  0.000000  0.000000  True


|1>       0.000000  0.250000  0.000000  0.000000  True


|+>       0.250000  0.000000  0.000000  0.000000  True


|->       0.000000  0.250000  0.000000  0.000000  True


|y+>      0.250000  0.250000  0.062500  0.062500  True


|y->      0.250000  0.250000  0.062500  0.062500  True


plein     0.250000  0.250000  0.062500  0.000000  False


  produit = 1/16 : True ; ecart a la borne = 1/16 : True



saturations : 6 registres purs sur 6 ; source : six sur sept, le plein exclus


### Lecture du résultat

Mesuré : la borne tient aux sept registres, avec **saturation aux six registres purs** — variances,
produits et bornes étant des rationnels exacts (aucune tolérance de comparaison), — au registre $|0\rangle$, l'éq. 57 de la source elle-même : $0 \cdot 1/4 \geq 0$ — et **pas au registre plein**, où les deux variances valent $1/4$, le produit $1/16$ et l'écart à la borne $1/16$, exactement les trois nombres du census de la source. Les registres saturés sont le noyau du défaut de Cauchy-Schwarz ; le registre plein n'en fait pas partie, et c'est précisément pour cela qu'il voit l'écart.

## 10. Ce que la source ne prouve pas — ligne à ligne

Le patron de la sous-série exige la liste, jamais maquillée en résultats établis :

1. **Les prémices ouvertes.** Les théorèmes 40.1 et 41.1 sont *conditionnels sur le texte de l'axiome et sur les résultats des chapitres nommés comme prémices, chacun à son propre rang* — et la chaîne des prémices, imprimée à la source, inclut les **résultats ouverts 2.4, 2.5, 2.7, 2.8, 18.5 et 18.6**. Tant que ces entrées restent ouvertes, les deux théorèmes restent conditionnels : ce carnet ne les déplace pas.
2. **Le lemme de monogamie** est *pris comme prémisse* dans la dérivation de la règle de Born, et **la borne de valence** *prise comme entrée* — aucun des deux n'est prouvé au chapitre 40.
3. **L'échelle du registre** (le pas du réseau en distillant $n$ copies) est *laissée ouverte* à la source.
4. **Le texte de l'axiome** est posé, pas dérivé — la phrase unique et les six définitions sont l'entrée du programme entier.
5. **Les correspondances nommées** (lemme de Schmidt [33], relation de Weyl [23], Gleason [70], envariance [71, 72], Robertson [73]) sont des mises en regard *par le nom* : la source dit elle-même qu'elles ne prouvent aucun théorème de la littérature.
6. **Le statut documentaire** : preprint non relu, *working draft* v0.1, produit d'une collaboration long terme ; les comptes du census (26 870 appariements, 64 triplets, douze pas, 96 / 2 304 / 84 éléments du groupe d'appui) sont déclarés par la source et certifiés à son dépôt, pas rejoués ici.

Ce que **ce carnet** ajoute de son côté : les deux sens de l'éq. 51, l'envariance, l'éq. 56, le join de la figure 15, la relation de Weyl, les recensements 3/24/84 et 3/6/21, et la borne aux sept registres — tous **Mesuré** ci-dessus. Rien de plus.

## 11. Croisements avec le reste du dépôt

- **`Physique/Chaines/` (Chaînes-01)** — même sous-série : l'état de paire $|\omega_m\rangle$, le plafond de lisibilité $d_1 d_2 \leq 6$ et la charge centrale y mènent au candidat $\alpha$ ; les chapitres 40-41 ici sont les prémices quantiques de ce chemin (l'espace d'états du chapitre 35 y est construit, ici cité).
- **`Physique/Jauge/` (Jauge-01 à 03)** — le croisement le plus direct : le compendium obtient **les lois de conservation et le groupe de jauge $SU(3) \times SU(2) \times U(1)$ comme théorèmes** (non conditionnels), quand Born, unitarité et incertitude restent conditionnels. Jauge-01 (Noether) et Jauge-02 (invariance de jauge) lisent le même bord physique depuis la symétrie, pas depuis l'information.
- **Lean** — 64 des 100 théorèmes de la release sont liés à des déclarations Lean nommées au dépôt `physical-superintelligence/chain-theory`, qui *vérifient les calculs finis* de leurs dérivations ; les chapitres 40-41 disent leurs objets « Certified at the repository; 30 objects enumerated ». Une distillation Lean d'un théorème conditionnel du compendium serait un grain candidat pour la série SymbolicAI/Lean.
- **Chapitre 42** (Measurement : the wash at one act and the POVM) — la suite immédiate du fil : le relevé collectif de la toile jointe, la POVM de la littérature. Non distillé ici ; candidat Chaînes-03.

## 12. Exercices

### Exercice 1 — le lien (3,3) et les deux sens de l'équivalence

Construisez la matrice de coefficients du ket de paire maximalement intriqué sur un lien $(3,3)$, puis vérifiez les deux sens de l'équation 51 : l'état réduit $CC^\dagger$ est plat, et la matrice est une unitaire à un facteur près ($C^\dagger C$ proportionnelle à l'identité). Le facteur vaut-il $1/3$ ?

In [10]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

// Exercice a completer
// Etape 1 : C = EtatPaireC02.PaireMaximale(3, 3)
// Etape 2 : calculer rho = C C^dagger et ses valeurs propres (attendu : plates)
// Etape 3 : calculer C^dagger C et verifier la proportionnalite a l'identite ; le facteur ?
object? resultat = null;  // TODO etudiant


### Exercice 2 — la troisième base du qubit

Au bout qubit, la troisième base du lien (celle de la lettre $XZ$) a pour vecteurs propres $|y+\rangle = (|0\rangle + i|1\rangle)/\sqrt 2$ et $|y-\rangle = (|0\rangle - i|1\rangle)/\sqrt 2$. Construisez l'observable d'étiquette $C$ de cette base (projecteur de rang un sur le vecteur étiqueté 1), vérifiez que $[A, C] \neq 0$ — le commutateur exact — puis que la borne de Robertson tient au registre plein pour la paire $(A, C)$.

In [11]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

// Exercice a completer
// Etape 1 : C = |y->><y-| (projecteur de rang un)
// Etape 2 : calculer [A, C] et verifier qu'il est non nul
// Etape 3 : variances de A et C au registre plein, esperance du commutateur, borne
object? resultat = null;  // TODO etudiant


### Exercice 3 — recompter les paires de bases à $d = 3$

À la dimension 3, notre énumération du §8 trouve 4 bases, donc 6 paires distinctes. Vérifiez-le à la main pour deux bases de votre choix : construisez leurs observables d'étiquette (les matrices diagonales dans leur base propre — pour la base du clock $Z$, c'est $\mathrm{diag}(0,1,2)$), calculez leur commutateur et vérifiez qu'il est non nul. Une base contre elle-même ?

In [12]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

// Exercice a completer
// Etape 1 : observable d'etiquette de la base de Z en d=3 : diag(0, 1, 2)
// Etape 2 : observable d'etiquette de la base de X : diagonaliser X en d=3, etiqueter 0..2
// Etape 3 : calculer le commutateur des deux observables, puis celui d'une base contre elle-meme
object? resultat = null;  // TODO etudiant


## À retenir

1. **Trois noms, un seul rang** : règle de Born, unitarité et relation d'incertitude sont, dans le compendium, des **théorèmes conditionnels** — conditionnels sur le texte de l'axiome et sur des prémices dont certaines restent **ouvertes**. La graduation (100 théorèmes, 109 conditionnels, 198 ouverts) est celle de la source ; ce carnet ne la déplace pas.
2. **Le pli plat est une équivalence** (éq. 51) : état réduit plat $\iff$ matrice de coefficients multiple d'une unitaire sur le support. L'unitarité du transport, du join et du pas en est le corollaire — le join de la figure 15 se rejoue comme un produit de matrices qui reste unitaire à un facteur près.
3. **La règle de Born comme unique relevé** : la trace du projecteur contre l'état est l'unique fonctionnel invariant sous le groupe d'appariement — dont l'état de paire est point fixe : $(U \otimes \bar U)|\omega\rangle = |\omega\rangle$ mesuré pour des unitaires aléatoires.
4. **L'incertitude depuis Weyl** : deux lectures du même lien dans deux bases ne commutent pas (éq. 55), et la borne de Robertson tient à tous les registres — saturée aux six registres purs, en écart de $1/16$ au registre plein (éq. 57). Nos recensements indépendants rendent les comptes de la source : 3/24/84 paires de lettres, 3/6/21 paires de bases.
5. **Ce qui reste ouvert** : les prémisses 2.4-2.8 et 18.5-18.6, le lemme de monogamie, la borne de valence, l'échelle du registre — et l'axiome lui-même, posé jamais dérivé. Le prochain grain du fil : le chapitre 42 (mesure, POVM).

## Références

**Source primaire** (archivée au gisement, licence CC BY 4.0, preprint non relu — *working draft*) :

- A. D. Wissner-Gross, *Chain Theory Compendium* (v0.1 working draft, 23/10/2026), chapitres 40 (« The Born rule and unitarity ») et 41 (« The uncertainty relation from the Weyl relation »), pp. 111-116 — `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2026 - Wissner-Gross - Chain Theory Compendium (v0.1 working draft).pdf`
- A. D. Wissner-Gross, *Chain Theory — The fine structure constant from a single axiom* (letter, 2026) — `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2026 - Wissner-Gross - Chain Theory - The fine structure constant from a single axiom (letter).pdf`
- Dépôt : [`physical-superintelligence/chain-theory`](https://github.com/physical-superintelligence/chain-theory) — les 64 déclarations Lean nommées vérifient les calculs finis des dérivations.

**Correspondances nommées par la source** (lues à la source, reproduites ici par le nom) : lemme de Schmidt [33] ; relation de Weyl [23] ; théorème de Gleason [70] ; envariance de Zurek [71, 72] ; forme de Robertson [73]. La source précise : ces mises en regard ne prouvent aucun théorème de la littérature.

**Jumeau** : [Chaînes-02 — version Python](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb) — mêmes
valeurs, arithmétique rationnelle exacte des deux côtés.

**Carnets voisins** : [Chaînes-01](Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb) (l'axiome, la charge centrale, le candidat $\alpha$) · [Jauge-01](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb) (Noether, conservation) · [Jauge-02](../Jauge/Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) (invariance de jauge $U(1) \times SU(2)$).

Issue de distillation : #25013 (Part of #25005).